In [ ]:
# Install the one dependency Kaggle's image lacks (torch/pandas/numpy/pyarrow/pyyaml already present)
!pip install higher==0.2.1 -q

In [ ]:
# Locate the uploaded bundle (auto-detect under /kaggle/input)
import os, glob

BUNDLE = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'checkpoint.pt' in files and 'panel.pkl' in files and 'args.yaml' in files:
        BUNDLE = root; break
assert BUNDLE, 'Bundle not found -- add the code+model dataset as an Input'
print('Bundle dir:', BUNDLE)
print('Contents:', sorted(os.listdir(BUNDLE)))

# Prefer the competition-provided test.parquet if present, else the bundle copy
comp_test = None
for p in glob.glob('/kaggle/input/**/test.parquet', recursive=True):
    if BUNDLE not in p:
        comp_test = p; break
TEST_PARQUET = comp_test or os.path.join(BUNDLE, 'test.parquet')
print('Using prices from:', TEST_PARQUET)

In [ ]:
# Generate submission.csv (loads pre-built panel + trained checkpoint; ~2 min, no big-RAM step)
cmd = (
    f'python "{BUNDLE}/code/forecasting_task/kaggle_submit.py" '
    f'--bundle "{BUNDLE}" '
    f'--test_parquet "{TEST_PARQUET}" '
    f'--out /kaggle/working/submission.csv'
)
print(cmd)
!{cmd}

In [ ]:
# Sanity-check output format (52,000 rows, [ID, Close])
import pandas as pd
sub = pd.read_csv('/kaggle/working/submission.csv')
print('shape:', sub.shape, '| columns:', list(sub.columns))
print(sub.head()); print(sub.tail())
assert list(sub.columns) == ['ID', 'Close']
assert len(sub) == 52000, f'expected 52000 rows, got {len(sub)}'
print('\nOK -- /kaggle/working/submission.csv ready to submit.')

### Training reproducibility (audit)

The checkpoint was trained with fixed seed 7 by `code/forecasting_task/run_DoubleAdapt.py` on the same competition data:

```
python code/forecasting_task/run_DoubleAdapt.py \
    --backbone dlinear --lr 0.002 --reg 1.0 \
    --use_target_novelty --use_related_novelty --news_weight_mult 10 \
    --freeze_online 1 --seed 7 \
    --data_dir <dir with all 6 *_textemb.parquet + test.parquet> \
    --logdir logs/dlinear_final_compliant
```

Re-running training needs all six embedding parquets in one folder and is memory-heavy (panel build peaks ~70GB RAM), so the submission path above loads the resulting checkpoint + pre-built panel instead. Both are derived only from competition-provided data (no external data/models).